Obtain ID's of records that have zero subjects.  
Then free to experiment with methods to enrich their metadata.

In [242]:
NO_RECORDS = 150

In [243]:
from datetime import datetime, timezone
from json import JSONDecodeError
from logging.config import dictConfig
from typing import Any, Optional
import psycopg
from psycopg import errors as psycopg_errors
from psycopg.rows import dict_row
from config.logging_config import LOGGING_CONFIG
from config.postgres_config import PostgresConfig
from tasks import transform_batch, add_file_metadata
import os
from fastapi import FastAPI, Query, HTTPException
import logging
from pydantic import BaseModel, Field
from utils.queue_utils import HarvestEventQueue, detect_identifier_type

from lxml import etree as ET
# API calls
import requests


# added for these experiments
from utils.enrichment_methods import get_doi, enrich_subjects_from_openaire

In [244]:
dictConfig(LOGGING_CONFIG)
logger = logging.getLogger(__name__)

In [245]:
user = os.getenv("POSTGRES_USER")
passwd = os.getenv("POSTGRES_PASSWORD")
host = os.getenv("POSTGRES_ADDRESS", "127.0.0.1")


if host == '':
    host = "127.0.0.1"
    user = os.getenv("POSTGRES_ADMIN")

connection_params = {
    "user" : user,
    "password" : passwd,
    "dbname" : "datasetdb",
    "port" : 5432,
    "host" : host
}

In [246]:
query_records = """
            -- Exploring subjects from record raw metadata

-- Every xpath defines a potential XML structure with corresponding subject tag placement.
-- If raw_metadata conforms to ANY of these xpaths, subjects will be concatenated <subject> tags. (removing array_to_string() will return arrays)
-- If raw_metadata doesn't conform to any path (and probably has no subject tags), subjects will be ''


WITH query AS (
    SELECT
        records.*,
        array_to_string(
                xpath('oai:record/oai:metadata/datacite:resource/datacite:subjects/datacite:subject/text()', raw_metadata,
                    '{{datacite, http://datacite.org/schema/kernel-4},
                      {oai, http://www.openarchives.org/OAI/2.0/}}'
                )::text[] ||
				xpath('oai:record/oai:metadata/oaidc:oai_datacite/oaidc:payload/datacite:resource/datacite:subjects/datacite:subject/text()', raw_metadata,
                    '{{datacite, http://datacite.org/schema/kernel-4},
                      {oai, http://www.openarchives.org/OAI/2.0/},
                      {oaidc, http://schema.datacite.org/oai/oai-1.1/}}'
                )::text[] ||
				xpath('oai:record/oai:metadata/oai:resource/datacite:subjects/datacite:subject/text()', raw_metadata,
                    '{{datacite, http://datacite.org/schema/kernel-4},
                      {oai, http://www.openarchives.org/OAI/2.0/}}'
                )::text[] ||
				xpath('oai:record/oai:metadata/oad:oai_datacite/oad:payload/datacite:resource/datacite:subjects/datacite:subject/text()', raw_metadata,
				-- note the 3 in datacite
                    '{{datacite, http://datacite.org/schema/kernel-3},
					  {oai, http://www.openarchives.org/OAI/2.0/},
     				  {oad, http://schema.datacite.org/oai/oai-1.0/}}'
                )::text[] ||
				-- same path as previous but uses kernel-4 namespace
				xpath('oai:record/oai:metadata/oad:oai_datacite/oad:payload/datacite:resource/datacite:subjects/datacite:subject/text()', raw_metadata,
                    '{{datacite, http://datacite.org/schema/kernel-4},
					  {oai, http://www.openarchives.org/OAI/2.0/},
     				  {oad, http://schema.datacite.org/oai/oai-1.0/}}'
                )::text[]
				, ', '
        ) AS subjects
    FROM public.records
)
SELECT 
    id,
    title,
    raw_metadata,
    subjects
FROM query
WHERE query.subjects = ''
LIMIT %s
;
"""

In [247]:
try:
    with psycopg.connect(**connection_params, row_factory=dict_row) as conn:
        cur = conn.cursor()
        
        cur.execute(query_records, (NO_RECORDS,))
        
        results = cur.fetchall()
        print(results)

        record_ids = [str(row["id"]) for row in results]
        record_titles = [str(row["title"]) for row in  results]
        record_raw_metadata = [str(row["raw_metadata"]) for row in  results]


except Exception as e:
    logger.exception("Failed to fetch harvest runs")
    raise HTTPException(status_code=500, detail=str(e))

[{'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1057341347', 'title': 'FInnish National Protein Crystallography Consortium', 'raw_metadata': '<record xmlns="http://www.openarchives.org/OAI/2.0/" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance">\n  <header>\n    <identifier>oai:icatplus.esrf.fr:inv/1057341347</identifier>\n    <datestamp>2023-06-19T06:00:14.182Z</datestamp>\n  </header>\n  <metadata>\n    <oai_datacite xmlns="http://schema.datacite.org/oai/oai-1.1/" xsi:schemaLocation="http://schema.datacite.org/oai/oai-1.1/ http://schema.datacite.org/oai/oai-1.1/oai.xsd">\n      <schemaVersion>4.2</schemaVersion>\n      <datacentreSymbol>EI</datacentreSymbol>\n      <payload>\n        <resource xmlns="http://schema.datacite.org/meta/kernel-4.2/" xsi:schemaLocation="http://schema.datacite.org/meta/kernel-4.2/ http://schema.datacite.org/meta/kernel-4.2/metadata.xsd"><identifier identifierType="DOI">10.15151/ESRF-ES-1057341347</identifier><creators><creator><creatorName nameType="

In [248]:
record_titles

['FInnish National Protein Crystallography Consortium',
 'Characterization of the microstructure of a modified solid-state electrolyte (LLZTO based).',
 'A high-resolution 3D map of manganese in the brain of a zebrafish model of manganese neurotoxicity.',
 'Analysing stacking disorder in KAg1-xAux(CN)2',
 'Connecting oxygen vacancy formation to the exsolution behavior of perovskite oxides',
 'NO BEAM - samples preparation for HC6338',
 'Misdirection of axonal outgrowth of nerve fibers after nerve injury and repair/reconstruction',
 'CHRONOS',
 'Holography Completion of MI-1384',
 'Macromolecular crystallography at the Philipps University Marburg (MarBAG)',
 'Disclosing Ce-In synergy and VO dynamic by XAS/XRD study on MOF-derived In2O3/CeZrOx catalyst for CO2-to-CH3OH conversion',
 'Structural Biology at EMBL Grenoble, IBS-CIBB, IAB and ILL',
 'PCT + DCT on In718 plastically deformed (~1%) - Comparison with CONFOCAL microscopy',
 'Structural Biology of Macromolecules and Macromolecular 

In [249]:
record_raw_metadata[2]

'<record xmlns="http://www.openarchives.org/OAI/2.0/" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance">\n  <header>\n    <identifier>oai:icatplus.esrf.fr:inv/1208264973</identifier>\n    <datestamp>2023-07-23T06:00:23.013Z</datestamp>\n  </header>\n  <metadata>\n    <oai_datacite xmlns="http://schema.datacite.org/oai/oai-1.1/" xsi:schemaLocation="http://schema.datacite.org/oai/oai-1.1/ http://schema.datacite.org/oai/oai-1.1/oai.xsd">\n      <schemaVersion>4.2</schemaVersion>\n      <datacentreSymbol>EI</datacentreSymbol>\n      <payload>\n        <resource xmlns="http://schema.datacite.org/meta/kernel-4.2/" xsi:schemaLocation="http://schema.datacite.org/meta/kernel-4.2/ http://schema.datacite.org/meta/kernel-4.2/metadata.xsd"><identifier identifierType="DOI">10.15151/ESRF-ES-1208264973</identifier><creators><creator><creatorName nameType="Personal">Marina ECKERMANN</creatorName><givenName>Marina</givenName><familyName>Eckermann</familyName><nameIdentifier nameIdentifierScheme="ORC

In [250]:
get_doi(record_raw_metadata[2])

'10.15151/ESRF-ES-1208264973'

# DOI check through bigger repositories  (#1)
Take an XML ETree.
Searches for identifier with identifierType DOI.
    - what if there are multiple, and one is under "related works"?
Seaches thru OpenAlex or Openaire - apis?
Data available? N -> end
Y ->
    search for fields: subject, topic, field, discipline, keyword

OpenAIRE API
- they're officialy moving to a Graph API in May 2026
https://api.openaire.eu/graph/swagger-ui/index.html

In [251]:
import logging
logging.getLogger("urllib3").setLevel(logging.WARNING)

In [252]:
doi = get_doi(record_raw_metadata[0]) 

if not doi:
    print("NO DOI")
else:
    print(enrich_subjects_from_openaire(doi))

['PaN', 'ID23-2', 'MX-2375', 'Particles, Nuclei and Fields', 'BM29', 'Macromolecular Crystallography', 'MX-2449']


In [253]:
# get only records that have a DOI
ix_has_doi = []

for i in range(0, len(record_raw_metadata)):
    rec = get_doi(record_raw_metadata[i])
    if rec:
        ix_has_doi.append(i)

print(ix_has_doi)

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149]


In [254]:
id_to_enrich = [record_ids[i] for i in ix_has_doi]
doi_to_enrich = [get_doi(record_raw_metadata[i]) for i in ix_has_doi]

print(id_to_enrich)
print(doi_to_enrich)

['5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1057341347', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2342430721', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1208264973', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2248514958', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2093751428', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2313091446', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/649330107', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2113174363', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/676113983', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/874803783', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2020218100', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2217188900', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1733308991', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/661300113', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/865155356', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2396842589', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/452932246', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/708312102'

In [255]:
# limit to smaller section
#id_to_enrich = id_to_enrich[:30]
#doi_to_enrich = doi_to_enrich[:30]

In [256]:
from utils.enrichment_methods import _openaire_batch_doi_resolve, _opeanaire_get_doi_subjects, _ids_connect_with_openaire_subjects, batch_enrich_subjects_from_openaire

api_result = _openaire_batch_doi_resolve(doi_to_enrich)

print(api_result[:3])

[{'authors': [{'fullName': 'Allen, Samuel', 'name': 'Samuel', 'surname': 'Allen', 'rank': 1, 'pid': None}, {'fullName': 'Capewell, Eve', 'name': 'Eve', 'surname': 'Capewell', 'rank': 2, 'pid': None}, {'fullName': 'Cassidy, Simon', 'name': 'Simon', 'surname': 'Cassidy', 'rank': 3, 'pid': {'id': {'scheme': 'orcid_pending', 'value': '0000-0002-4297-1425'}, 'provenance': None}}], 'openAccessColor': None, 'publiclyFunded': None, 'type': 'dataset', 'language': {'code': 'eng', 'label': 'English'}, 'countries': None, 'subjects': [{'subject': {'scheme': 'keyword', 'value': 'ID22'}, 'provenance': None}, {'subject': {'scheme': 'keyword', 'value': 'Chemistry'}, 'provenance': None}, {'subject': {'scheme': 'keyword', 'value': 'CH-7620'}, 'provenance': None}], 'mainTitle': 'Analysing stacking disorder in KAg1-xAux(CN)2', 'subTitle': None, 'descriptions': ['The structures of a potassium- dicyanoargentate and dicyanoaurate solid solution are to be measured as a function of temperature in order to model

In [257]:
doi_subjects = _opeanaire_get_doi_subjects(api_result)
doi_subjects

[{'doi': '10.15151/esrf-es-2248514958',
  'subjects': ['ID22', 'Chemistry', 'CH-7620']},
 {'doi': '10.15151/esrf-es-2313091446',
  'subjects': ['BM32',
   'IH-HC-4245',
   'In-house Hard Condensed Matter Science']},
 {'doi': '10.15151/esrf-es-2112610773',
  'subjects': ['PaN',
   'Earth Sciences',
   'x-ray probe',
   'Particles, Nuclei and Fields',
   'BM05',
   'BM18',
   'ES-1503']},
 {'doi': '10.15151/esrf-es-2113174295',
  'subjects': ['PaN',
   'Earth Sciences',
   'x-ray probe',
   'Particles, Nuclei and Fields',
   'BM05',
   'BM18',
   'ES-1503']},
 {'doi': '10.15151/esrf-es-2113174363',
  'subjects': ['PaN',
   'Earth Sciences',
   'x-ray probe',
   'Particles, Nuclei and Fields',
   'BM05',
   'BM18',
   'ES-1503']},
 {'doi': '10.15151/esrf-es-2112563831',
  'subjects': ['PaN',
   'Earth Sciences',
   'x-ray probe',
   'Particles, Nuclei and Fields',
   'BM05',
   'BM18',
   'ES-1503']},
 {'doi': '10.15151/esrf-es-2113708104',
  'subjects': ['PaN',
   'Earth Sciences',
   'x

In [258]:
id_dois = []
for ix in range(len(id_to_enrich)):
    id_dois.append(
        {
            'id': id_to_enrich[ix],
            'doi': doi_to_enrich[ix].lower(),
        }
    )
id_dois

[{'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1057341347',
  'doi': '10.15151/esrf-es-1057341347'},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2342430721',
  'doi': '10.15151/esrf-es-2342430721'},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1208264973',
  'doi': '10.15151/esrf-es-1208264973'},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2248514958',
  'doi': '10.15151/esrf-es-2248514958'},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2093751428',
  'doi': '10.15151/esrf-es-2093751428'},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2313091446',
  'doi': '10.15151/esrf-es-2313091446'},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/649330107',
  'doi': '10.15151/esrf-es-649330107'},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2113174363',
  'doi': '10.15151/esrf-es-2113174363'},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/676113983',
  'doi': '10.15151/esrf-es-676113983'},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/87480378

In [259]:
enriched_list = _ids_connect_with_openaire_subjects(id_dois, doi_subjects)
enriched_list

[{'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1057341347',
  'doi': '10.15151/esrf-es-1057341347',
  'enriched_subjects': {'BM29',
   'ID23-2',
   'MX-2375',
   'MX-2449',
   'Macromolecular Crystallography',
   'PaN',
   'Particles, Nuclei and Fields'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2342430721',
  'doi': '10.15151/esrf-es-2342430721',
  'enriched_subjects': {'ID13', 'IH-MA-740', 'IH-MA-744', 'IH-MA-796'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1208264973',
  'doi': '10.15151/esrf-es-1208264973',
  'enriched_subjects': {'ID16A',
   'LS-3191',
   'Life Sciences',
   'PaN',
   'Particles, Nuclei and Fields'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2248514958',
  'doi': '10.15151/esrf-es-2248514958',
  'enriched_subjects': {'CH-7620', 'Chemistry', 'ID22'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2093751428',
  'doi': '10.15151/esrf-es-2093751428',
  'enriched_subjects': {'Applied Material Science',
   'ID15A',
   'MA-6542',


In [260]:
# Only succesfully enriched IDs and subjects.
# Could be useful for comparing how many ids we sent to be enriched vs
# how many actually got new subjects
[el for el in enriched_list if el['enriched_subjects'] is not None]

[{'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1057341347',
  'doi': '10.15151/esrf-es-1057341347',
  'enriched_subjects': {'BM29',
   'ID23-2',
   'MX-2375',
   'MX-2449',
   'Macromolecular Crystallography',
   'PaN',
   'Particles, Nuclei and Fields'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2342430721',
  'doi': '10.15151/esrf-es-2342430721',
  'enriched_subjects': {'ID13', 'IH-MA-740', 'IH-MA-744', 'IH-MA-796'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1208264973',
  'doi': '10.15151/esrf-es-1208264973',
  'enriched_subjects': {'ID16A',
   'LS-3191',
   'Life Sciences',
   'PaN',
   'Particles, Nuclei and Fields'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2248514958',
  'doi': '10.15151/esrf-es-2248514958',
  'enriched_subjects': {'CH-7620', 'Chemistry', 'ID22'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2093751428',
  'doi': '10.15151/esrf-es-2093751428',
  'enriched_subjects': {'Applied Material Science',
   'ID15A',
   'MA-6542',


In [261]:
# All-in-one function

batch_enrich_subjects_from_openaire(id_dois)

[{'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1057341347',
  'doi': '10.15151/esrf-es-1057341347',
  'enriched_subjects': {'BM29',
   'ID23-2',
   'MX-2375',
   'MX-2449',
   'Macromolecular Crystallography',
   'PaN',
   'Particles, Nuclei and Fields'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2342430721',
  'doi': '10.15151/esrf-es-2342430721',
  'enriched_subjects': {'ID13', 'IH-MA-740', 'IH-MA-744', 'IH-MA-796'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1208264973',
  'doi': '10.15151/esrf-es-1208264973',
  'enriched_subjects': {'ID16A',
   'LS-3191',
   'Life Sciences',
   'PaN',
   'Particles, Nuclei and Fields'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2248514958',
  'doi': '10.15151/esrf-es-2248514958',
  'enriched_subjects': {'CH-7620', 'Chemistry', 'ID22'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2093751428',
  'doi': '10.15151/esrf-es-2093751428',
  'enriched_subjects': {'Applied Material Science',
   'ID15A',
   'MA-6542',


In [262]:
# Comparison: Number of initial records vs number of records that have gained subjects via this method
enriched = batch_enrich_subjects_from_openaire(id_dois)
print(len([el['doi'] for el in enriched]))
print(len([el['doi'] for el in enriched if el['enriched_subjects'] is not None]))

150
150


In [263]:
# Taking a look at the metadata initially obtained from DB, subset of few records
full_info_enriched = [el for el in enriched if el['enriched_subjects'] is not None]

first_enriched = full_info_enriched[:10]

In [264]:
# indexes that have been enriched
ix_original = [record_ids.index(rec['id']) for rec in first_enriched]
ix_original

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]

In [265]:
first_enriched

[{'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1057341347',
  'doi': '10.15151/esrf-es-1057341347',
  'enriched_subjects': {'BM29',
   'ID23-2',
   'MX-2375',
   'MX-2449',
   'Macromolecular Crystallography',
   'PaN',
   'Particles, Nuclei and Fields'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2342430721',
  'doi': '10.15151/esrf-es-2342430721',
  'enriched_subjects': {'ID13', 'IH-MA-740', 'IH-MA-744', 'IH-MA-796'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1208264973',
  'doi': '10.15151/esrf-es-1208264973',
  'enriched_subjects': {'ID16A',
   'LS-3191',
   'Life Sciences',
   'PaN',
   'Particles, Nuclei and Fields'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2248514958',
  'doi': '10.15151/esrf-es-2248514958',
  'enriched_subjects': {'CH-7620', 'Chemistry', 'ID22'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2093751428',
  'doi': '10.15151/esrf-es-2093751428',
  'enriched_subjects': {'Applied Material Science',
   'ID15A',
   'MA-6542',


In [266]:
[record_raw_metadata[i] for i in ix_original]

['<record xmlns="http://www.openarchives.org/OAI/2.0/" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance">\n  <header>\n    <identifier>oai:icatplus.esrf.fr:inv/1057341347</identifier>\n    <datestamp>2023-06-19T06:00:14.182Z</datestamp>\n  </header>\n  <metadata>\n    <oai_datacite xmlns="http://schema.datacite.org/oai/oai-1.1/" xsi:schemaLocation="http://schema.datacite.org/oai/oai-1.1/ http://schema.datacite.org/oai/oai-1.1/oai.xsd">\n      <schemaVersion>4.2</schemaVersion>\n      <datacentreSymbol>EI</datacentreSymbol>\n      <payload>\n        <resource xmlns="http://schema.datacite.org/meta/kernel-4.2/" xsi:schemaLocation="http://schema.datacite.org/meta/kernel-4.2/ http://schema.datacite.org/meta/kernel-4.2/metadata.xsd"><identifier identifierType="DOI">10.15151/ESRF-ES-1057341347</identifier><creators><creator><creatorName nameType="Personal">Dihia MOUSSAOUI</creatorName><givenName>Dihia</givenName><familyName>Moussaoui</familyName><nameIdentifier nameIdentifierScheme="ORCI

## OpenAlex API
https://developers.openalex.org/
https://developers.openalex.org/guides/authentication

Get multiple works by DOI

use | between DOIs, up to 100 DOIs at once!!
check out how the results look then.

https://api.openalex.org/works?filter=doi:https://doi.org/10.1234/a|https://doi.org/10.1234/b|https://doi.org/10.1234/c&per_page=100



Use the select parameter to return only specific fields:
https://api.openalex.org/works?select=id,doi,display_name

In [267]:
OpenAlex_API_key = os.getenv("OPENALEX_API_KEY")

In [268]:
OPENALEX_API_URL = "https://api.openalex.org/works/"

doi = '10.15151/esrf-es-1473312981'

# We have to append the ID directly in the path
query_url = OPENALEX_API_URL + "https://doi.org/" + doi

response = requests.get(
    query_url,
    params={"api_key": OpenAlex_API_key}, 
    timeout=30
)

data = response.json()


In [269]:
data

{'id': 'https://openalex.org/W6886229830',
 'doi': 'https://doi.org/10.15151/esrf-es-1473312981',
 'title': 'SAXS WAXS measurements on reference samples for different sample detector distances. Control of transmission measurements',
 'display_name': 'SAXS WAXS measurements on reference samples for different sample detector distances. Control of transmission measurements',
 'publication_year': 2024,
 'publication_date': '2024-01-25',
 'ids': {'openalex': 'https://openalex.org/W6886229830',
  'doi': 'https://doi.org/10.15151/esrf-es-1473312981'},
 'language': 'en',
 'primary_location': {'id': 'doi:10.15151/esrf-es-1473312981',
  'is_oa': True,
  'landing_page_url': 'https://doi.org/10.15151/esrf-es-1473312981',
  'pdf_url': None,
  'source': {'id': 'https://openalex.org/S7407051764',
   'display_name': 'European Synchrotron Radiation Facility',
   'issn_l': None,
   'issn': [],
   'is_oa': False,
   'is_in_doaj': False,
   'is_core': False,
   'host_organization': None,
   'host_organiza

In [270]:
oa_DOIS = [
    '10.15151/esrf-es-1001518064',
    '10.15151/esrf-es-2323561777',
    '10.15151/esrf-es-2323561778',
    '10.15151/esrf-es-2323561877',
    'https://doi.org/10.5286/ISIS.E.100051344',
    '10.34894/AUUSS2',
    '10.17026/DANS-28H-C53V',
    '10.34894/VLSN1K',
    '10.15151/ESRF-ES-1030397492',
    '10.17026/AR/DMHU53',
    '10.17026/AR/GHYQTO',
    '10.17026/DANS-ZU2-HF3N',
    '10.17026/DANS-ZTM-7SA5',
    '10.17026/DANS-ZUK-HN2Z',
    'https://doi.org/10.1016/j.paid.2021.110888',
    '10.23662/fors-ds-686-1'
]

In [271]:
OPENALEX_API_URL = "https://api.openalex.org/works"

def _openalex_batch_doi_resolve(dois, api_key):
    all_results = []
    chunk_size = 100

    # Strip any existing https://doi.org/ or http://doi.org/ prefix
    # so we can add it back consistently for every DOI
    clean_dois = [
        doi.replace("https://doi.org/", "")
           .replace("http://doi.org/", "")
        for doi in dois
    ]

    for i in range(0, len(clean_dois), chunk_size):
        chunk = clean_dois[i : i + chunk_size]

        # Join with pipe — each DOI gets the full https://doi.org/ prefix
        doi_filter = "|".join(f"https://doi.org/{doi}" for doi in chunk)
        # Build URL as f-string to avoid requests encoding the colons and pipes
        url = f"{OPENALEX_API_URL}?filter=doi:{doi_filter}&per_page={chunk_size}&api_key={api_key}"

        response = requests.get(url, timeout=30)
        response.raise_for_status()
        all_results.extend(response.json().get("results", []))

    return all_results

In [272]:
results = _openalex_batch_doi_resolve(oa_DOIS,
                    api_key=OpenAlex_API_key)
print(len(results))
results

2


[{'id': 'https://openalex.org/W3152931024',
  'doi': 'https://doi.org/10.1016/j.paid.2021.110888',
  'title': 'Personality, anxiety, and cognitive failures as predictors of flow proneness',
  'display_name': 'Personality, anxiety, and cognitive failures as predictors of flow proneness',
  'publication_year': 2021,
  'publication_date': '2021-04-08',
  'ids': {'openalex': 'https://openalex.org/W3152931024',
   'doi': 'https://doi.org/10.1016/j.paid.2021.110888',
   'mag': '3152931024'},
  'language': 'en',
  'primary_location': {'id': 'doi:10.1016/j.paid.2021.110888',
   'is_oa': False,
   'landing_page_url': 'https://doi.org/10.1016/j.paid.2021.110888',
   'pdf_url': None,
   'source': {'id': 'https://openalex.org/S151331055',
    'display_name': 'Personality and Individual Differences',
    'issn_l': '0191-8869',
    'issn': ['0191-8869', '1873-3549'],
    'is_oa': False,
    'is_in_doaj': False,
    'is_core': True,
    'host_organization': 'https://openalex.org/P4310320990',
    'ho

In [273]:
# Get potential subjects, two relevant fields: keywords and concepts
# Concepts is an older system, keywords is newer - but concepts is more diverse

In [274]:
kw_lists = [rez['keywords'] for rez in results]

keywords = [[item['display_name'] for item in sublist] for sublist in kw_lists]

keywords

[['Psychology',
  'Conscientiousness',
  'Extraversion and introversion',
  'Neuroticism',
  'Openness to experience',
  'Context (archaeology)',
  'Anxiety',
  'Personality',
  'Big Five personality traits',
  'Developmental psychology',
  'Clinical psychology',
  'Social psychology',
  'Psychiatry'],
 ['Cell',
  'Extracellular matrix',
  'Pathogenesis',
  'Protein folding',
  'Function (biology)',
  'Enzyme',
  'Cell adhesion',
  'Folding (DSP implementation)',
  'Gene']]

In [275]:
# Introduce a threshold of score:
keywords = [[item['display_name'] for item in sublist if item['score'] > 0.6] for sublist in kw_lists]

keywords

[['Psychology',
  'Conscientiousness',
  'Extraversion and introversion',
  'Neuroticism',
  'Openness to experience'],
 []]

In [276]:
def _opeanalex_get_doi_subjects(response: list[dict], score_treshold: float = 0.5):
    """
    Transform API response into a list of DOI-sets and their subject-sets.

    A single OpenAIRE ID will result in one or more DOIs and zero or more subjects.
    If is has no linked subjects, we can't enrich anything so we skip that ID entirely.
    The OpenAIRE ID itself doesn't concern us so it is not retained. 
    
    :param response: List[dict], the API response
    :return: List[dict]
        dict:
            DOI - DOI that is linked to at least one subject
            [Subjects] - all subjects for the given DOI
    """
    if response == []:
        # no DOIs with any results
        return []
    
    # sets of DOIs with relevant sets of subjects. Later, spread to individual DOIs
    enriched_dois = []
    enriched_subjects = []

    for el in response:
        kw_list = el['keywords']

        if (kw_list is None or kw_list == []):
            # this element doesn't contribute to enriching subjects,
            # so even the DOIs don't matter
            continue

        keywords = [item['display_name'] for item in kw_list if item['score'] > score_treshold]

        # After filtering by score it's possible that no keywords are left,
        # Then this DOI won't be enriched
        if not keywords:
            continue
        
        current_doi = el['doi']

        # Strip it from prefixes and make it lowercase
        current_doi = current_doi.replace("https://doi.org/", "").replace("http://doi.org/", "").lower()

        enriched_dois.append(current_doi)

        enriched_subjects.append(keywords)

    # Create a list of dicts as output

    doi_subjects = []

    for ix in range(len(enriched_dois)):
        doi_sub_pair = {'doi': enriched_dois[ix],
                        'subjects': enriched_subjects[ix]}
        doi_subjects.append(doi_sub_pair)

    return doi_subjects

In [277]:
_opeanalex_get_doi_subjects(results)

[{'doi': '10.1016/j.paid.2021.110888',
  'subjects': ['Psychology',
   'Conscientiousness',
   'Extraversion and introversion',
   'Neuroticism',
   'Openness to experience',
   'Context (archaeology)',
   'Anxiety',
   'Personality',
   'Big Five personality traits']},
 {'doi': '10.15151/esrf-es-2323561777',
  'subjects': ['Cell',
   'Extracellular matrix',
   'Pathogenesis',
   'Protein folding',
   'Function (biology)']}]

In [278]:
_opeanalex_get_doi_subjects(results, score_treshold=0.65)

[{'doi': '10.1016/j.paid.2021.110888',
  'subjects': ['Psychology',
   'Conscientiousness',
   'Extraversion and introversion',
   'Neuroticism',
   'Openness to experience']}]

In [279]:
_opeanalex_get_doi_subjects(results, score_treshold=0)

[{'doi': '10.1016/j.paid.2021.110888',
  'subjects': ['Psychology',
   'Conscientiousness',
   'Extraversion and introversion',
   'Neuroticism',
   'Openness to experience',
   'Context (archaeology)',
   'Anxiety',
   'Personality',
   'Big Five personality traits',
   'Developmental psychology',
   'Clinical psychology',
   'Social psychology',
   'Psychiatry']},
 {'doi': '10.15151/esrf-es-2323561777',
  'subjects': ['Cell',
   'Extracellular matrix',
   'Pathogenesis',
   'Protein folding',
   'Function (biology)',
   'Enzyme',
   'Cell adhesion',
   'Folding (DSP implementation)',
   'Gene']}]

In [280]:
id_dois

[{'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1057341347',
  'doi': '10.15151/esrf-es-1057341347'},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2342430721',
  'doi': '10.15151/esrf-es-2342430721'},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1208264973',
  'doi': '10.15151/esrf-es-1208264973'},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2248514958',
  'doi': '10.15151/esrf-es-2248514958'},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2093751428',
  'doi': '10.15151/esrf-es-2093751428'},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2313091446',
  'doi': '10.15151/esrf-es-2313091446'},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/649330107',
  'doi': '10.15151/esrf-es-649330107'},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2113174363',
  'doi': '10.15151/esrf-es-2113174363'},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/676113983',
  'doi': '10.15151/esrf-es-676113983'},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/87480378

In [281]:
res = _openalex_batch_doi_resolve([el['doi'] for el in id_dois],
                            OpenAlex_API_key)

subj = _opeanalex_get_doi_subjects(res, score_treshold=0)

subj

[{'doi': '10.15151/esrf-es-2232202253',
  'subjects': ['Upgrade',
   'State (computer science)',
   'Process (computing)',
   'Block (permutation group theory)',
   'Solid-state']},
 {'doi': '10.15151/esrf-es-2243238189',
  'subjects': ['Catalysis',
   'Cationic polymerization',
   'Transition metal',
   'Decomposition',
   'Thermal decomposition',
   'Hydrogen',
   'Hydrogen production',
   'Oxygen']},
 {'doi': '10.15151/esrf-es-2237123105',
  'subjects': ['Diamond',
   'Synchrotron',
   'Doping',
   'Germanium',
   'Silicon',
   'Diamond cubic',
   'Graphite',
   'Hexagonal crystal system']},
 {'doi': '10.15151/esrf-es-2248514958',
  'subjects': ['Stacking',
   'Supercell',
   'Diffraction',
   'Hexagonal crystal system',
   'Rietveld refinement',
   'Function (biology)',
   'Thermal']},
 {'doi': '10.15151/esrf-es-2248514697',
  'subjects': ['Raman scattering',
   'Synchrotron',
   'Raman spectroscopy',
   'Scattering',
   'Synchrotron radiation',
   'Biogeochemical cycle']},
 {'doi'

In [282]:
len(res)

21

In [283]:
connected = _ids_connect_with_openaire_subjects(id_dois, subj)
connected

[{'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1057341347',
  'doi': '10.15151/esrf-es-1057341347',
  'enriched_subjects': None},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2342430721',
  'doi': '10.15151/esrf-es-2342430721',
  'enriched_subjects': {'Characterization (materials science)',
   'Dendrite (mathematics)',
   'Electrolyte',
   'Microstructure',
   'Strain (injury)'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1208264973',
  'doi': '10.15151/esrf-es-1208264973',
  'enriched_subjects': None},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2248514958',
  'doi': '10.15151/esrf-es-2248514958',
  'enriched_subjects': {'Diffraction',
   'Function (biology)',
   'Hexagonal crystal system',
   'Rietveld refinement',
   'Stacking',
   'Supercell',
   'Thermal'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2093751428',
  'doi': '10.15151/esrf-es-2093751428',
  'enriched_subjects': None},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2313091446',
  

In [284]:
successfully_enriched = [el for el in connected if el['enriched_subjects'] is not None]
print(len(successfully_enriched))
successfully_enriched

21


[{'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2342430721',
  'doi': '10.15151/esrf-es-2342430721',
  'enriched_subjects': {'Characterization (materials science)',
   'Dendrite (mathematics)',
   'Electrolyte',
   'Microstructure',
   'Strain (injury)'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2248514958',
  'doi': '10.15151/esrf-es-2248514958',
  'enriched_subjects': {'Diffraction',
   'Function (biology)',
   'Hexagonal crystal system',
   'Rietveld refinement',
   'Stacking',
   'Supercell',
   'Thermal'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2313091446',
  'doi': '10.15151/esrf-es-2313091446',
  'enriched_subjects': {'Beam (structure)',
   'Cathode ray',
   'Layer (electronics)',
   'Process (computing)'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2396842589',
  'doi': '10.15151/esrf-es-2396842589',
  'enriched_subjects': {'Battery (electricity)',
   'Power (physics)',
   'Process (computing)',
   'Work (physics)'}},
 {'id': '5d1a3aed-114c-4

In [285]:
len(id_dois)

150

In [286]:
enr_doi = [el['doi'] for el in successfully_enriched]

res_doi = [el['doi'] for el in res]

subj_doi = [el['doi'] for el in subj]

In [287]:
print(len(enr_doi))
print(len(res_doi))

21
21


In [288]:
[el for el in res if el['doi'].replace("https://doi.org/", "") not in enr_doi]

[]

In [289]:
input_dois = {el['doi'] for el in id_dois}
result_dois = {el['doi'] for el in subj}

print(len(result_dois - input_dois))
result_dois - input_dois

0


set()

In [290]:
print(len(input_dois - result_dois))
input_dois - result_dois

129


{'10.15151/esrf-es-1001518064',
 '10.15151/esrf-es-1004811608',
 '10.15151/esrf-es-1018399818',
 '10.15151/esrf-es-1021965520',
 '10.15151/esrf-es-1021966178',
 '10.15151/esrf-es-1025992787',
 '10.15151/esrf-es-1026318791',
 '10.15151/esrf-es-1038848224',
 '10.15151/esrf-es-1041475936',
 '10.15151/esrf-es-1057341347',
 '10.15151/esrf-es-1081709123',
 '10.15151/esrf-es-1103065461',
 '10.15151/esrf-es-1118675452',
 '10.15151/esrf-es-1120155666',
 '10.15151/esrf-es-117583838',
 '10.15151/esrf-es-1178035195',
 '10.15151/esrf-es-118169974',
 '10.15151/esrf-es-118686985',
 '10.15151/esrf-es-1191058575',
 '10.15151/esrf-es-1205127635',
 '10.15151/esrf-es-1208264973',
 '10.15151/esrf-es-1262206544',
 '10.15151/esrf-es-128944986',
 '10.15151/esrf-es-1299132129',
 '10.15151/esrf-es-1299177199',
 '10.15151/esrf-es-1304325403',
 '10.15151/esrf-es-1317773810',
 '10.15151/esrf-es-1324918289',
 '10.15151/esrf-es-1329162612',
 '10.15151/esrf-es-1330738188',
 '10.15151/esrf-es-1338641718',
 '10.15151/e

In [291]:
api_result_dois = {el['doi'].replace("https://doi.org/", "").lower() for el in res}
api_result_dois

{'10.15151/esrf-es-2232202253',
 '10.15151/esrf-es-2233771865',
 '10.15151/esrf-es-2237123105',
 '10.15151/esrf-es-2243038067',
 '10.15151/esrf-es-2243238189',
 '10.15151/esrf-es-2248514697',
 '10.15151/esrf-es-2248514958',
 '10.15151/esrf-es-2286110233',
 '10.15151/esrf-es-2287254195',
 '10.15151/esrf-es-2298273021',
 '10.15151/esrf-es-2312892293',
 '10.15151/esrf-es-2313091446',
 '10.15151/esrf-es-2313092331',
 '10.15151/esrf-es-2313109812',
 '10.15151/esrf-es-2317715311',
 '10.15151/esrf-es-2323561172',
 '10.15151/esrf-es-2323561777',
 '10.15151/esrf-es-2342430721',
 '10.15151/esrf-es-2348957236',
 '10.15151/esrf-es-2356892770',
 '10.15151/esrf-es-2396842589'}

In [292]:
api_result_dois - input_dois

set()

In [293]:
res_not_enriching = api_result_dois - result_dois
res_not_enriching

set()

In [294]:
res_ne_info = [el for el in res if el['doi'].replace("https://doi.org/", "").lower() in res_not_enriching]
len(res_ne_info)

0

In [295]:
[el['keywords'] for el in res_ne_info]

[]

the API result object can have a result for a given DOI, but that result doesn't contain any keywords = doesn't contribute to enrichment
Hence the len(api_result) can be larger than len(enriched_dois)

Additionaly if we set a high keyword score threshold, a certain result MIGHT have keywords but none are relevant enough - that will also not contribute to enrichment

# DOI resolver method  (#2)
Take an XML ETree.
Searches for identifier with identifierType DOI.
    - what if there are multiple, and one is under "related works"?
Talks to an API such as DataCite or SignPosting.
Data available? N -> end
Y ->
    search for fields: subject, topic, field, discipline, keyword